In [20]:
!pip -q install --upgrade \
  "pandas==2.2.2" \
  "pyarrow==17.0.0" \
  "datasets==3.0.1" \
  "fsspec==2024.6.1" \
  "gcsfs==2024.6.1" \
  "transformers" \
  "accelerate" \
  "safetensors"


In [21]:
# Re-run this cell
!pip -q install --upgrade transformers datasets accelerate safetensors

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.


In [22]:
# If your model lives in Google Drive, mount it:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
!find /content -maxdepth 3 -type d -name "medical-report-generator-final" 2>/dev/null || true
!find /content/drive/MyDrive -maxdepth 5 -type d -name "medical-report-generator-final" 2>/dev/null || true


/content/drive/MyDrive/medical-report-generator-final
/content/drive/MyDrive/medical-report-generator-final


In [24]:
import csv
import json

# This template creates the instruction for the model
PROMPT_TEMPLATE = """Generate a clinical summary for the {section_header} section based on the following conversation.

Conversation:
\"\"\"
{dialogue}
\"\"\"

Summary:"""

def create_fine_tuning_data(input_csv_path, output_jsonl_path):
    """
    Reads a CSV file and converts it to a JSON Lines (.jsonl) file
    for fine-tuning.
    """
    try:
        with open(input_csv_path, mode='r', encoding='utf-8') as csv_file, \
             open(output_jsonl_path, mode='w', encoding='utf-8') as jsonl_file:

            csv_reader = csv.DictReader(csv_file)

            processed_count = 0
            for row in csv_reader:
                # Format the prompt using the data from the CSV
                prompt = PROMPT_TEMPLATE.format(
                    section_header=row.get('section_header', '').strip(),
                    dialogue=row.get('dialogue', '').strip()
                )

                # The 'completion' is the target summary the model should learn
                completion = row.get('section_text', '').strip()

                # Write the prompt-completion pair as a JSON object on a new line
                jsonl_file.write(json.dumps({"prompt": prompt, "completion": completion}) + "\n")
                processed_count += 1

        print(f"✅ Successfully created '{output_jsonl_path}' with {processed_count} examples.")

    except FileNotFoundError:
        print(f"❌ Error: The file '{input_csv_path}' was not found. Make sure you've uploaded it.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")


# --- Main execution ---
training_input = 'MTS-Dialog-TrainingSet.csv'
training_output = 'training_data.jsonl'

validation_input = 'MTS-Dialog-ValidationSet.csv'
validation_output = 'validation_data.jsonl'

print("--- Starting Data Preparation ---")
create_fine_tuning_data(training_input, training_output)
create_fine_tuning_data(validation_input, validation_output)
print("--- Data preparation complete. ---")

--- Starting Data Preparation ---
✅ Successfully created 'training_data.jsonl' with 1201 examples.
✅ Successfully created 'validation_data.jsonl' with 100 examples.
--- Data preparation complete. ---


In [25]:
!pip install --upgrade -q transformers datasets torch accelerate evaluate rouge_score matplotlib

In [29]:
# === Mount Drive so your saved model persists across sessions ===
from google.colab import drive
drive.mount('/content/drive')

# === Install/upgrade libs (safe to re-run) ===
!pip -q install --upgrade transformers datasets accelerate safetensors

# === TRAINING SCRIPT (Simplified & Colab-ready) ===
import json
import os
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
import torch
from pathlib import Path

# -------------------
# Configuration
# -------------------
BASE_MODEL_NAME = "gpt2"
DATASET_NAME = "har1/MTS_Dialogue-Clinical_Note"

# Save FINAL model to Drive so it persists
OUTPUT_MODEL_DIR = "/content/drive/MyDrive/medical-report-generator-final"
Path(OUTPUT_MODEL_DIR).mkdir(parents=True, exist_ok=True)

PROMPT_TEMPLATE = """Generate a clinical summary for the {section_header} section based on the following conversation.

Conversation:
\"\"\"
{dialogue}
\"\"\"

Summary:"""

print(f"Loading dataset '{DATASET_NAME}'...")
dataset = load_dataset(DATASET_NAME)

# Use a small validation split (optional)
train_test_split = dataset['train'].train_test_split(test_size=0.1, seed=42)
train_dataset = train_test_split['train']

print(f"Loading model and tokenizer for '{BASE_MODEL_NAME}'...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=True)

# GPT-2 has no pad token by default — map pad -> eos to avoid warnings & enable batching
if tokenizer.pad_token is None and tokenizer.eos_token is not None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME)

def format_and_tokenize(examples):
    # Builds: [PROMPT][TARGET_TEXT] as a single LM training sequence
    full_text = []
    for header, dialogue, text in zip(examples["section_header"], examples["dialogue"], examples["section_text"]):
        prompt = PROMPT_TEMPLATE.format(section_header=header, dialogue=dialogue)
        full_text.append(prompt + text)
    # Keep sequences reasonable; GPT-2 context is ~1024 tokens
    return tokenizer(full_text, truncation=True, padding="max_length", max_length=512)

print("Tokenizing the dataset...")
tokenized_train_dataset = train_dataset.map(
    format_and_tokenize,
    batched=True,
    remove_columns=train_dataset.column_names
)

# Data collator will create labels by shifting input_ids for causal LM
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# You can tweak these depending on your GPU/CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

training_args = TrainingArguments(
    output_dir="/content/tmp-medical-run",  # working dir for checkpoints/logs (runtime)
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    learning_rate=5e-5,
    warmup_ratio=0.03,
    weight_decay=0.0,
    logging_steps=50,
    save_strategy="epoch",      # save checkpoint each epoch (in /content/tmp-medical-run)
    report_to="none",
    fp16=torch.cuda.is_available(),  # mixed precision on GPU
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    data_collator=data_collator,
)

print("\n🚀 Starting fine-tuning...")
trainer.train()
print("✅ Training complete.")

# ADD THIS BLOCK TO SAVE THE LOGS ===============================
import json
import os

# Explicitly save the log history to your final directory
log_history = trainer.state.log_history
log_file_path = os.path.join(OUTPUT_MODEL_DIR, "training_logs.json")

print(f"💾 Saving training log history to: {log_file_path}")
with open(log_file_path, "w") as f:
    json.dump(log_history, f)
# ===============================================================

# -------------------
# Save FINAL model + tokenizer to Drive
# -------------------
print(f"💾 Saving final model + tokenizer to: {OUTPUT_MODEL_DIR}")
model.save_pretrained(OUTPUT_MODEL_DIR, safe_serialization=True)  # writes model.safetensors + config.json
tokenizer.save_pretrained(OUTPUT_MODEL_DIR)                        # writes tokenizer files
print("✅ Saved.")

# -------------------
# (Optional) Quick reload sanity check
# -------------------
from transformers import AutoTokenizer, AutoModelForCausalLM
re_tok = AutoTokenizer.from_pretrained(OUTPUT_MODEL_DIR, local_files_only=True)
re_model = AutoModelForCausalLM.from_pretrained(OUTPUT_MODEL_DIR, local_files_only=True)
print("🔁 Reload test OK:", re_tok.pad_token_id is not None and hasattr(re_model, "generate"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading dataset 'har1/MTS_Dialogue-Clinical_Note'...
Loading model and tokenizer for 'gpt2'...
Tokenizing the dataset...


Map:   0%|          | 0/1170 [00:00<?, ? examples/s]

Device: cuda

🚀 Starting fine-tuning...


Step,Training Loss
50,2.557800
100,1.805600
150,1.672500
200,1.683700
250,1.599800
300,1.500500
350,1.528900
400,1.437700
450,1.461900
500,1.484000


✅ Training complete.
💾 Saving training log history to: /content/drive/MyDrive/medical-report-generator-final/training_logs.json
💾 Saving final model + tokenizer to: /content/drive/MyDrive/medical-report-generator-final
✅ Saved.
🔁 Reload test OK: True


In [31]:
# Run this cell to check the contents of your output directory
!ls -l /content/drive/MyDrive/medical-report-generator-final

total 490814
-rw------- 1 root root       874 Sep 29 20:23 config.json
-rw------- 1 root root       119 Sep 29 20:23 generation_config.json
-rw------- 1 root root    456318 Sep 29 20:23 merges.txt
-rw------- 1 root root 497774208 Sep 29 20:23 model.safetensors
-rw------- 1 root root       131 Sep 29 20:23 special_tokens_map.json
-rw------- 1 root root       507 Sep 29 20:23 tokenizer_config.json
-rw------- 1 root root   3557957 Sep 29 20:23 tokenizer.json
-rw------- 1 root root      2428 Sep 29 20:23 training_logs.json
-rw------- 1 root root    798156 Sep 29 20:23 vocab.json


In [33]:
#
# SEPARATE SCRIPT FOR EVALUATION
#

# Install evaluation libraries if needed
!pip install -q evaluate rouge_score

import torch
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

# --- Configuration ---
# CORRECTED: Point to the final model directory on your Google Drive
MODEL_PATH = '/content/drive/MyDrive/medical-report-generator-final'
DATASET_NAME = "har1/MTS_Dialogue-Clinical_Note"
PROMPT_TEMPLATE = """Generate a clinical summary for the {section_header} section based on the following conversation.

Conversation:
\"\"\"
{dialogue}
\"\"\"

Summary:"""

# --- Load Data and Model ---
print("Loading test data...")
# We reload and split the data with the same seed to ensure we get the identical test set
dataset = load_dataset(DATASET_NAME)
train_test_split = dataset['train'].train_test_split(test_size=0.1, seed=42)
test_dataset = train_test_split['test']

print(f"Loading your fine-tuned model from '{MODEL_PATH}'...")
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH).to(device)

# --- Generate Predictions ---
print("Generating predictions on the test set...")
predictions = []
references = []

for example in tqdm(test_dataset):
    prompt = PROMPT_TEMPLATE.format(section_header=example["section_header"], dialogue=example["dialogue"])
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    output = model.generate(
        **inputs,
        max_new_tokens=150,
        num_beams=5,
        early_stopping=True
    )

    prediction_text = tokenizer.decode(output[0], skip_special_tokens=True)
    summary = prediction_text.split("Summary:")[-1].strip()

    predictions.append(summary)
    references.append(example["section_text"])

# --- Compute and Print ROUGE Scores ---
print("Computing ROUGE scores...")
rouge = evaluate.load("rouge")
results = rouge.compute(predictions=predictions, references=references)

print("\n" + "="*50)
print("      FINAL EVALUATION METRICS")
print("="*50)
for key, value in results.items():
    print(f"{key.capitalize():<10}: {value*100:.2f}") # Print as percentage
print("="*50)

Loading test data...
Loading your fine-tuned model from '/content/drive/MyDrive/medical-report-generator-final'...
Generating predictions on the test set...


100%|██████████| 131/131 [04:12<00:00,  1.93s/it]


Computing ROUGE scores...



      FINAL EVALUATION METRICS
Rouge1    : 62.93
Rouge2    : 55.01
Rougel    : 62.10
Rougelsum : 62.39


Rouge1: Measures the overlap of individual words to check for content relevance.

Rouge2: Measures the overlap of word pairs to check for phrasal fluency.

Rougel: Measures the longest common sequence of words to check for sentence-level similarity.

Rougelsum: Measures the longest common sequence of words at the whole summary level.


In [34]:
# ==============================================================================
# SCRIPT FOR INTERACTIVE REPORT GENERATION
# ==============================================================================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import os
import sys

# --- Configuration ---
# This path must point to your final model directory in Google Drive
MODEL_PATH = "/content/drive/MyDrive/medical-report-generator-final"

PROMPT_TEMPLATE = """Generate a clinical summary for the {section_header} section based on the following conversation.

Conversation:
\"\"\"
{dialogue}
\"\"\"

Summary:"""

# --- Load Model and Tokenizer ---
print(f"--- Loading Model from {MODEL_PATH} ---")

# Check if the model directory exists before proceeding
if not os.path.isdir(MODEL_PATH):
    print(f"❌ Error: Model directory not found at '{MODEL_PATH}'")
    print("Please make sure you have successfully run the training script and the path is correct.")
    # Stop execution if the model can't be found
    sys.exit()

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH).to(device)
print(f"✅ Model loaded successfully on device: {device}")


def generate_summary(section_header, dialogue):
    """Generates a clinical summary using the fine-tuned model."""
    prompt = PROMPT_TEMPLATE.format(section_header=section_header, dialogue=dialogue)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    output = model.generate(
        **inputs,
        max_new_tokens=200,      # Max length of the summary
        num_beams=5,             # Use beam search for higher quality output
        no_repeat_ngram_size=2,  # Prevent repeating phrases
        early_stopping=True
    )

    # Decode the output and extract only the summary part
    summary = tokenizer.decode(output[0], skip_special_tokens=True).split("Summary:")[-1].strip()
    return summary

# --- Interactive Loop ---
while True:
    print("\n" + "="*50)
    print("      New Medical Report Generation")
    print("="*50)

    try:
        # Get user input
        section = input("Enter the Section Header (e.g., CC, ROS) or 'exit' to quit: ")
        if section.lower() in ['quit', 'exit']:
            break

        print("Enter the Dialogue (type 'END' on a new line to finish):")
        dialogue_lines = []
        while True:
            line = input()
            if line.strip().upper() == 'END':
                break
            dialogue_lines.append(line)
        dialogue = "\n".join(dialogue_lines)

        if not dialogue.strip():
            print("❌ No dialogue entered. Please try again.")
            continue

        # Generate and print the summary
        print("\n--- Generating Summary... ---")
        generated_summary = generate_summary(section, dialogue)

        print("\n" + "-"*50)
        print("✅ AI-Generated Clinical Summary:")
        print(generated_summary)
        print("-"*50)

    except KeyboardInterrupt:
        # Allow exiting with Ctrl+C
        break
    except Exception as e:
        print(f"An error occurred: {e}")
        break

print("\nExiting program. Goodbye!")

--- Loading Model from /content/drive/MyDrive/medical-report-generator-final ---
✅ Model loaded successfully on device: cuda

      New Medical Report Generation
Enter the Section Header (e.g., CC, ROS) or 'exit' to quit: CC
Enter the Dialogue (type 'END' on a new line to finish):
Doctor: "Good morning, Ms. Davis. So, what brings you in to see me today?"  Patient: "Hi, Doctor. I've been having this really bad headache for the past two days."  Doctor: "I'm sorry to hear that. Can you show me where it hurts?"  Patient: "It's mainly on the right side of my head, right behind my eye."  Doctor: "And on a scale of 1 to 10, with 10 being the worst, how would you rate the pain?"  Patient: "It's been a pretty constant 7 out of 10. It's a sharp, throbbing pain."  Doctor: "Okay, thank you. Is the headache the main reason for your visit?"  Patient: "Yes, that's the only thing that's bothering me right now."
END


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



--- Generating Summary... ---

--------------------------------------------------
✅ AI-Generated Clinical Summary:
Symptoms: N/A
Diagnosis: Headache, headache on right-hand side, pain on left-sided area, rated as 7/10 on scale 10-10 with 7 being worst headache in the history of the patient, no other known causes of headache, patient denies any other symptoms, denies pain related to headache or pain in eye, notes that headache is the primary cause of headaches, does not have any known neurological or psychiatric side effects, has no known medical history, and denies the possibility of any neurological, psychiatric, or other medical conditions that could be associated with this headache. 
History of Complaint: Symptoms started about two weeks ago and worsened over the last 2 days. Patient has been experiencing headaches on both sides of his head since the onset of symptoms. He has not been able to control his headaches or control the intensity or duration of these headaches. Symptoms ha

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



--- Generating Summary... ---

--------------------------------------------------
✅ AI-Generated Clinical Summary:
Symptoms: N/A
Diagnosis: Ankle injury, pain in the inside of the right ankle, difficulty in lifting weight, worsening pain if attempting weight lifting, history of ankle sprains, no other significant medical issues, ankle pain is rated as moderate to severe by the PASTSURGICAL ASSESSMENT section, patient is wearing ankle brace and ankle bandage and is not wearing any other type of protective clothing, does not have any significant other medical problems, the patient has not been diagnosed with any major medical conditions, has been referred to a specialist for evaluation and treatment, denies pain or discomfort in ankle or ankle replacement surgery, did not injure himself or others in any way during the course of his visit, correctable ankle injury was diagnosed as mild to moderate in nature and was not related to any specific medical condition, was referred by an orthope

In [1]:
# Install dependencies (run once)
!pip install datasets transformers pandas tqdm --quiet

from datasets import load_dataset
from transformers import pipeline
import pandas as pd
from tqdm import tqdm

# Load MTS Dialog dataset from Hugging Face
dataset = load_dataset("mts-dataset/mts-dialog")

# You can choose the split (train/test/validation)
dialogs = dataset["train"]

# Inspect available columns
print("Columns:", dialogs.column_names)

# Assuming the dialogue text is stored under 'dialog' or 'dialogue'
# If unsure, print a few samples
print(dialogs[0])

# Initialize summarization model
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

# Generate reports (summaries) for each dialogue
reports = []
for d in tqdm(dialogs, desc="Generating reports"):
    text = d.get("dialog", d.get("dialogue", ""))  # handle column name variations
    if not text:
        reports.append("No dialogue found")
        continue
    
    try:
        summary = summarizer(text, max_length=120, min_length=30, do_sample=False)[0]['summary_text']
        reports.append(summary)
    except Exception as e:
        reports.append(f"Error: {e}")

# Create DataFrame and save to CSV
df = pd.DataFrame({
    "dialogue": [d.get("dialog", d.get("dialogue", "")) for d in dialogs],
    "report": reports
})

df.to_csv("mts_dialog_reports.csv", index=False)
print("✅ Report saved to mts_dialog_reports.csv")



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip
c:\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetNotFoundError: Dataset 'mts-dataset/mts-dialog' doesn't exist on the Hub or cannot be accessed.